# 🧠 Deep Learning Neural Networks for Salary Prediction

## Overview
This notebook explores deep learning approaches for Ethiopian salary prediction using neural networks. We'll implement various architectures and compare them with traditional ML models.

### Neural Network Architectures:
- 🔗 **Multi-Layer Perceptron (MLP)**: Basic feedforward networks
- 🎯 **Deep Neural Networks**: Multiple hidden layers
- 🔄 **Regularized Networks**: Dropout and batch normalization
- 🎨 **Custom Architectures**: Domain-specific designs
- ⚡ **Optimized Training**: Advanced optimizers and learning rates
- 📊 **Ensemble Methods**: Combining multiple neural networks

### Advanced Techniques:
- 🎛️ **Hyperparameter Tuning**: Grid search and Bayesian optimization
- 📈 **Learning Curves**: Training progress visualization
- 🔍 **Feature Importance**: Neural network interpretability
- 🎯 **Cross-Validation**: Robust model evaluation

---

In [ ]:
# Import comprehensive deep learning libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

# TensorFlow/Keras for deep learning
try:
    import tensorflow as tf
    from tensorflow import keras
    from tensorflow.keras import layers, models, optimizers, callbacks
    from tensorflow.keras.utils import plot_model
    from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
    TENSORFLOW_AVAILABLE = True
    print(f"✅ TensorFlow {tf.__version__} available")
    
    # Set random seeds for reproducibility
    tf.random.set_seed(42)
    
except ImportError:
    TENSORFLOW_AVAILABLE = False
    print("⚠️ TensorFlow not available. Install with: pip install tensorflow")

# PyTorch alternative
try:
    import torch
    import torch.nn as nn
    import torch.optim as optim
    from torch.utils.data import DataLoader, TensorDataset
    PYTORCH_AVAILABLE = True
    print(f"✅ PyTorch {torch.__version__} available")
except ImportError:
    PYTORCH_AVAILABLE = False
    print("⚠️ PyTorch not available. Install with: pip install torch")

# Scikit-learn for preprocessing and comparison
from sklearn.model_selection import train_test_split, KFold
from sklearn.preprocessing import StandardScaler, OneHotEncoder, LabelEncoder
from sklearn.compose import ColumnTransformer
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.neural_network import MLPRegressor

# Advanced optimization
try:
    from keras_tuner import RandomSearch, BayesianOptimization
    KERAS_TUNER_AVAILABLE = True
    print("✅ Keras Tuner available for hyperparameter optimization")
except ImportError:
    KERAS_TUNER_AVAILABLE = False
    print("⚠️ Keras Tuner not available. Install with: pip install keras-tuner")

# Visualization and utilities
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from IPython.display import display, HTML, SVG
import time
import joblib
from datetime import datetime

# Set random seeds for reproducibility
np.random.seed(42)

print("\n🧠 Deep Learning Setup Complete!")
print(f"🔥 TensorFlow/Keras: {TENSORFLOW_AVAILABLE}")
print(f"⚡ PyTorch: {PYTORCH_AVAILABLE}")
print(f"🎛️ Keras Tuner: {KERAS_TUNER_AVAILABLE}")
print("🚀 Ready for Advanced Neural Network Training!")

In [ ]:
# Load and prepare data for deep learning
print("📥 PREPARING DATA FOR DEEP LEARNING")
print("=" * 50)

# Load dataset
df = pd.read_csv('../ethiopia_salary_data.csv')
print(f"✅ Dataset loaded: {df.shape[0]} rows × {df.shape[1]} columns")

# Handle missing values
df['test_score'].fillna(df['test_score'].median(), inplace=True)
print(f"✅ Missing values handled")

# Prepare features and target
feature_columns = ['experience_years', 'test_score', 'department', 'education_level', 'location']
target_column = 'salary_etb'

X = df[feature_columns].copy()
y = df[target_column].copy()

# Split data
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f"📊 Training samples: {X_train.shape[0]}")
print(f"📊 Testing samples: {X_test.shape[0]}")

# Advanced preprocessing for neural networks
print("\n🔧 Advanced Preprocessing for Neural Networks")
print("-" * 45)

# Separate numerical and categorical features
numerical_features = ['experience_years', 'test_score']
categorical_features = ['department', 'education_level', 'location']

# Create preprocessing pipeline
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numerical_features),
        ('cat', OneHotEncoder(drop='first', sparse_output=False), categorical_features)
    ]
)

# Fit and transform the data
X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

# Scale target variable for better neural network training
target_scaler = StandardScaler()
y_train_scaled = target_scaler.fit_transform(y_train.values.reshape(-1, 1)).flatten()
y_test_scaled = target_scaler.transform(y_test.values.reshape(-1, 1)).flatten()

print(f"✅ Features processed: {X_train_processed.shape[1]} dimensions")
print(f"✅ Target variable scaled")

# Get feature names for interpretation
feature_names = (numerical_features + 
                list(preprocessor.named_transformers_['cat'].get_feature_names_out(categorical_features)))

print(f"\n🏷️ Feature names ({len(feature_names)} total):")
for i, name in enumerate(feature_names[:10]):
    print(f"   {i+1:2d}. {name}")
if len(feature_names) > 10:
    print(f"   ... and {len(feature_names) - 10} more features")

# Convert to appropriate format for deep learning
if TENSORFLOW_AVAILABLE:
    X_train_tf = tf.constant(X_train_processed, dtype=tf.float32)
    X_test_tf = tf.constant(X_test_processed, dtype=tf.float32)
    y_train_tf = tf.constant(y_train_scaled, dtype=tf.float32)
    y_test_tf = tf.constant(y_test_scaled, dtype=tf.float32)
    print(f"✅ TensorFlow tensors created")

if PYTORCH_AVAILABLE:
    X_train_torch = torch.FloatTensor(X_train_processed)
    X_test_torch = torch.FloatTensor(X_test_processed)
    y_train_torch = torch.FloatTensor(y_train_scaled)
    y_test_torch = torch.FloatTensor(y_test_scaled)
    print(f"✅ PyTorch tensors created")

print("\n🎯 Data preparation complete! Ready for neural network training.")